# GemCol Evaluation: Phase 7 (Analytics & Failure Case Analysis)
This notebook analyzes the results from the MS MARCO validation runs to generate metrics for the Discussion section of the paper.

In [ ]:
import json
import random
from collections import Counter
import sys

# Using the beir loader to get qrels, queries, and corpus if needed,
# or relying on the local JSON runs to do the breakdown.
print("Loading Run Files...")
with open('/workspace/results/bm25_run.json', 'r') as f:
    bm25_run = json.load(f)
with open('/workspace/results/bge_run.json', 'r') as f:
    bge_run = json.load(f)
with open('/workspace/results/colbert_run.json', 'r') as f:
    colbert_run = json.load(f)

# We also need qrels. Assuming they are saved or can be loaded via HuggingFace/BeIR
from datasets import load_dataset
qrels_ds = load_dataset("BeIR/msmarco-qrels", split="test")
qrels = {}
for row in qrels_ds:
    qid = str(row['query-id'])
    docid = str(row['corpus-id'])
    score = row['score']
    if qid not in qrels:
        qrels[qid] = {}
    qrels[qid][docid] = score

queries_ds = load_dataset("Tevatron/msmarco-passage", "default", split="dev")
queries = {row['query_id']: row['query'] for row in queries_ds}


## 1. Retrieval Source Breakdown
For all queries where TriRank successfully placed the relevant document in the Top 10, where did that document originally come from? Was it uniquely found by BM25, uniquely by BGE, or both?

In [ ]:
source_stats = Counter()

for qid, colbert_docs in colbert_run.items():
    if qid not in qrels:
        continue
    
    relevant_docs = qrels[qid]
    top_10 = colbert_docs[:10]
    
    # Check if a relevant doc is in top 10
    found_rel_doc = None
    for doc_id in top_10:
        if doc_id in relevant_docs and relevant_docs[doc_id] > 0:
            found_rel_doc = doc_id
            break
            
    if found_rel_doc:
        in_bm25 = found_rel_doc in bm25_run.get(qid, [])[:100]
        in_bge = found_rel_doc in bge_run.get(qid, [])[:100]
        
        if in_bm25 and in_bge:
            source_stats['Both'] += 1
        elif in_bm25:
            source_stats['BM25_Only'] += 1
        elif in_bge:
            source_stats['BGE_Only'] += 1
        else:
            source_stats['Unknown'] += 1

total_success = sum(source_stats.values())
print("=== Retrieval Source Breakdown (Top 10 Successes) ===")
for source, count in source_stats.items():
    percentage = (count / total_success) * 100
    print(f"{source}: {count} ({percentage:.2f}%)")

## 2. Qualitative Failure Case Analysis
Find queries where TriRank failed to place the relevant document in the Top 10.

In [ ]:
failures = []
for qid, colbert_docs in colbert_run.items():
    if qid not in qrels:
        continue
    relevant_docs = [doc_id for doc_id, score in qrels[qid].items() if score > 0]
    if not relevant_docs:
        continue
    rel_doc = relevant_docs[0]
    
    top_10 = colbert_docs[:10]
    if rel_doc not in top_10:
        failures.append((qid, rel_doc))

print(f"Found {len(failures)} failure cases out of {len(colbert_run)} evaluated queries.\n")

random.seed(42)
sample_failures = random.sample(failures, min(3, len(failures)))

for i, (qid, rel_doc) in enumerate(sample_failures):
    print(f"\n{'='*50}")
    print(f"Failure Case {i+1} (Query ID: {qid})")
    print(f"Query: {queries.get(qid, 'Unknown')}")
    print(f"Relevant Doc ID: {rel_doc}")
    
    # Check where it failed
    in_bm25 = rel_doc in bm25_run.get(qid, [])[:100]
    in_bge = rel_doc in bge_run.get(qid, [])[:100]
    colbert_rank = colbert_run[qid].index(rel_doc) + 1 if rel_doc in colbert_run[qid] else ">100 (Not in candidate pool)"
    
    print("\nDiagnostics:")
    print(f"- Was it in BM25 Top 100? {in_bm25}")
    print(f"- Was it in BGE Top 100? {in_bge}")
    print(f"- Final ColBERT Rank: {colbert_rank}")
    print('='*50)
